# 02 · Abbreviation expansions — `RC → ROMAN CATHOLIC`, `HMP → HIS MAJESTYS PRISON`

**Feature request:** *"feat: further expansions/abbreviations"* — e.g.
`RC CHURCH → ROMAN CATHOLIC CHURCH` and `HMP → His Majesty's Prison`.

This notebook explains **where** expansions live, **how** they are applied, **why**
these cases matter, and the one architectural constraint a contributor must respect.
It uses only tiny in-memory data, so it runs offline.

## Where expansions are defined

A single JSON file, shipped inside the package:

`uk_address_matcher/data/address_abbreviations.json` — a list of
`{"token": <abbrev>, "replacement": <expansion>}` objects (e.g. `RD → ROAD`,
`FLT → FLAT`). Adding `RC` and `HMP` here is the entire contribution surface for data;
no code change is required.

In [1]:
import json
import importlib.resources as ir
import duckdb

raw = ir.files("uk_address_matcher.data").joinpath("address_abbreviations.json").read_text()
abbr = json.loads(raw)
tokens = {d["token"].upper() for d in abbr}

print(f"{len(abbr)} abbreviation entries currently shipped")
print("RC present? ", "RC" in tokens)
print("HMP present?", "HMP" in tokens)
print("\nA few existing entries:")
print([d for d in abbr if d["token"] in ("RC", "HMP")])

147 abbreviation entries currently shipped
RC present?  False
HMP present? False

A few existing entries:
[]


## How expansions are applied — **single-token MAP lookup** (the key constraint)

In `cleaning/steps/normalisation.py`, `_normalise_abbreviations_and_units()` builds a
DuckDB `MAP` from the JSON and applies it **token by token**:

```sql
array_to_string(
  list_transform(
    string_split(clean_full_address, ' '),     -- split on spaces
    x -> COALESCE(map_extract(abbr_map, x)[1], x)  -- replace token if found, else keep
  ), ' ')
```

Consequences you must design around:

- The **key** is matched against a *single whitespace token*. A multi-word key like
  `CAR PARK SPACE` can **never** match this map.
- The **replacement value may be multiple words** — `array_to_string` re-joins them.
  So `RC → ROMAN CATHOLIC` turns `RC CHURCH` into `ROMAN CATHOLIC CHURCH` for free.
- Matching is case-insensitive (keys are upper-cased) and runs **after** punctuation
  is stripped — so encode `His Majesty's` as `HIS MAJESTYS` (no apostrophe).

### See the *real* pipeline expand an existing abbreviation

`prepare_data_for_matching` runs the full cleaning pipeline and exposes the
`clean_full_address` column. Watch `FLT`/`GRD`/`RD` expand:

In [2]:
import duckdb
import pyarrow as pa

from uk_address_matcher.cleaning.chunking_strategies import prepare_data_for_matching

UK_POSTCODE_REGEX = r"\b(?:GIR ?0AA|[A-Z][A-HJ-Y]?\d[A-Z\d]? ?\d[A-Z]{2})\b"


messy = [
    {
        "unique_id": "a",
        "address_concat": "FLAT D1 2ND FLR EXAMPLE HOUSE LONDON",
        "postcode": "SE1 2AB",
        "ukam_label": "1",
    },
    {
        "unique_id": "b",
        "address_concat": "UNIT Q3 2ND FLR EXAMPLE WORKS LONDON",
        "postcode": "SE1 2AB",
        "ukam_label": "2",
    },
    {
        "unique_id": "c",
        "address_concat": "FLAT A 3RD FLR EXAMPLE HOUSE LONDON",
        "postcode": "SE1 2AB",
        "ukam_label": "3",
    },
]
messy_pa = pa.Table.from_pylist(messy)

con = duckdb.connect()
con.register("messy_df", messy_pa)




In [7]:
messy

[{'unique_id': 'a',
  'address_concat': 'FLAT D1 2ND FLR EXAMPLE HOUSE LONDON',
  'postcode': 'SE1 2AB',
  'ukam_label': '1'},
 {'unique_id': 'b',
  'address_concat': 'UNIT Q3 2ND FLR EXAMPLE WORKS LONDON',
  'postcode': 'SE1 2AB',
  'ukam_label': '2'},
 {'unique_id': 'c',
  'address_concat': 'FLAT A 3RD FLR EXAMPLE HOUSE LONDON',
  'postcode': 'SE1 2AB',
  'ukam_label': '3'}]

In [3]:
cleaned = prepare_data_for_matching(
    con.table("messy_df"),
    con=con,
    num_of_chunks=1,
    dataset_role="messy",
)

repro = cleaned.project(
    f"""
    unique_id,
    original_address_concat,
    postcode,
    regexp_extract(
        upper(original_address_concat),
        '{UK_POSTCODE_REGEX}'
    ) as postcode_like_substring,
    clean_full_address,
    flat_positional,
    flat_identity,
    numeric_tokens,
    regexp_matches(clean_full_address, '\\b2ND\\b') as has_2nd,
    regexp_matches(clean_full_address, '\\bSECOND\\b') as has_second,
    regexp_matches(clean_full_address, '\\b3RD\\b') as has_3rd,
    regexp_matches(clean_full_address, '\\bTHIRD\\b') as has_third
    """
)


in_rows = pa.Table.from_pylist(messy)
duckdb.sql("select * from in_rows").show(max_width=100000)


repro.show(max_width=100000)

┌───────────┬──────────────────────────────────────┬──────────┬────────────┐
│ unique_id │            address_concat            │ postcode │ ukam_label │
│  varchar  │               varchar                │ varchar  │  varchar   │
├───────────┼──────────────────────────────────────┼──────────┼────────────┤
│ a         │ FLAT D1 2ND FLR EXAMPLE HOUSE LONDON │ SE1 2AB  │ 1          │
│ b         │ UNIT Q3 2ND FLR EXAMPLE WORKS LONDON │ SE1 2AB  │ 2          │
│ c         │ FLAT A 3RD FLR EXAMPLE HOUSE LONDON  │ SE1 2AB  │ 3          │
└───────────┴──────────────────────────────────────┴──────────┴────────────┘

┌───────────┬──────────────────────────────────────┬──────────┬─────────────────────────┬────────────────────────────────────────────┬─────────────────┬─────────────────┬────────────────┬─────────┬────────────┬─────────┬───────────┐
│ unique_id │       original_address_concat        │ postcode │ postcode_like_substring │             clean_full_address             │ flat_positional

In [12]:
from uk_address_matcher import prepare_data_for_matching

con = duckdb.connect(":memory:")
demo = con.sql("""
SELECT * FROM (VALUES
  ('a1', 'FLT 5 GRD FLOOR 10 HIGH RD', 'AB1 2CD'),
  ('a2', 'APT 2 SUMMER LN', 'AB1 2CD')
) t(unique_id, address_concat, postcode)
""")
prepare_data_for_matching(demo, con=con).select(
    "original_address_concat, clean_full_address"
).df()

,original_address_concat,clean_full_address
0,FLT 5 GRD FLOOR 10 HIGH RD,FLAT 5 GROUND FLOOR 10 HIGH ROAD
1,APT 2 SUMMER LN,APARTMENT 2 SUMMER LANE


## Simulating the proposed `RC` / `HMP` additions

We can't edit the installed package's JSON from here (the upstream clone must stay
pristine), so we **replicate the exact single-token MAP transform** in DuckDB with the
shipped abbreviations *plus* the two proposed entries. This is precisely what the
pipeline step would produce after the JSON change.

In [ ]:
def test_rc_and_hmp_abbreviation_expansion(duck_con):
    """RC -> ROMAN CATHOLIC and HMP -> HIS MAJESTYS PRISON (issue #365).

    Both are single-token keys whose replacement expands to multiple words. The
    surrounding tokens (CHURCH, PRESBYTERY, ARMLEY, ...) must be left untouched, and the
    expansion must fire wherever the token appears, not only at the start of the string.
    """
    input_rel = duck_con.sql(
        """
        SELECT * FROM (VALUES
            ('RC CHURCH 5 EXAMPLE ROAD LONDON'),
            ('ST MARYS RC PRIMARY SCHOOL CHURCH LANE'),
            ('FLAT 2 RC PRESBYTERY 9 CHAPEL STREET'),
            ('HMP LEEDS ARMLEY'),
            ('HMP WORMWOOD SCRUBS 160 DU CANE ROAD LONDON')
        ) AS t(clean_full_address)
    """
    )

    pipeline = create_sql_pipeline(
        con=duck_con,
        input_rel=input_rel,
        stage_specs=[_normalise_abbreviations_and_units],
    )
    result_rel = pipeline.run()
    clean_idx = result_rel.columns.index("clean_full_address")
    actual = [row[clean_idx] for row in result_rel.fetchall()]

    assert actual == [
        "ROMAN CATHOLIC CHURCH 5 EXAMPLE ROAD LONDON",
        "ST MARYS ROMAN CATHOLIC PRIMARY SCHOOL CHURCH LANE",
        "FLAT 2 ROMAN CATHOLIC PRESBYTERY 9 CHAPEL STREET",
        "HIS MAJESTYS PRISON LEEDS ARMLEY",
        "HIS MAJESTYS PRISON WORMWOOD SCRUBS 160 DU CANE ROAD LONDON",
    ]


In [13]:
import pandas as pd

PROPOSED = [
    {"token": "RC",  "replacement": "ROMAN CATHOLIC"},
    {"token": "HMP", "replacement": "HIS MAJESTYS PRISON"},   # apostrophe already stripped upstream
]
abbr_df = pd.DataFrame(abbr, columns=["token", "replacement"])
con.register("abbr", abbr_df)

# Cleaned text is upper cased + punctuation-free by the time abbreviation expansion runs.
test_inputs = [
    "RC CHURCH HALL 5 EXAMPLE ROAD",
    "HMP LEEDS ARMLEY",
    "FLAT 2 RC PRESBYTERY 9 CHAPEL STREET",
]
inp_values = ", ".join(f"('{s}')" for s in test_inputs)

con.sql(f"""
WITH lookup AS (
    SELECT UPPER(TRIM(token)) AS token, TRIM(replacement) AS replacement
    FROM abbr WHERE token IS NOT NULL AND replacement IS NOT NULL
),
m AS (SELECT map(list(token), list(replacement)) AS abbr_map FROM lookup),
inp AS (SELECT * FROM (VALUES {inp_values}) t(clean_full_address))
SELECT
  inp.clean_full_address AS before,
  array_to_string(
    list_transform(string_split(inp.clean_full_address, ' '),
                   x -> COALESCE(map_extract(m.abbr_map, x)[1], x)),
    ' '
  ) AS after
FROM inp CROSS JOIN m
""").df()

,before,after
0,RC CHURCH HALL 5 EXAMPLE ROAD,ROMAN CATHOLIC CHURCH HALL 5 EXAMPLE ROAD
1,HMP LEEDS ARMLEY,HIS MAJESTYS PRISON LEEDS ARMLEY
2,FLAT 2 RC PRESBYTERY 9 CHAPEL STREET,FLAT 2 ROMAN CATHOLIC PRESBYTERY 9 CHAPEL STREET


In [5]:
import pandas as pd

PROPOSED = [
    {"token": "RC",  "replacement": "ROMAN CATHOLIC"},
    {"token": "HMP", "replacement": "HIS MAJESTYS PRISON"},   # apostrophe already stripped upstream
]

expansion_df = pd.DataFrame(abbr + PROPOSED, columns=["token", "replacement"])
con.register("expansion", expansion_df)

# Cleaned text is upper cased + punctuation free by the time abbreviation expansion runs. 
test_inputs = [
    "RC CHURCH HALL 5 EXAMPLE ROAD",
    "HMP Birmingham Winson Green Road Birmingham B18 4AS",
    "HMP Onley Rugby Warwickshire CV23 8AP",
    "FLAT 2 RC PRESBYTERY 9 CHAPEL STREET",
]

inp_values = ", ".join(f"('{s}')" for s in test_inputs)

con.sql(f"""
WITH lookup AS (
    SELECT UPPER(TRIM(token)) AS token, TRIM(replacement) AS replacement
    FROM abbr WHERE token IS NOT NULL AND replacement IS NOT NULL
),
m AS (SELECT map(list(token), list(replacement)) AS abbr_map FROM lookup),
inp AS (SELECT * FROM (VALUES {inp_values}) t(clean_full_address))
SELECT
  inp.clean_full_address AS before,
  array_to_string(
    list_transform(string_split(inp.clean_full_address, ' '),
                   x -> COALESCE(map_extract(m.abbr_map, x)[1], x)),
    ' '
  ) AS after
FROM inp CROSS JOIN m
""").df()

,before,after
0,RC CHURCH HALL 5 EXAMPLE ROAD,ROMAN CATHOLIC CHURCH HALL 5 EXAMPLE ROAD
1,HMP Birmingham Winson Green Road Birmingham B1...,HIS MAJESTYS PRISON Birmingham Winson Green Ro...
2,HMP Onley Rugby Warwickshire CV23 8AP,HIS MAJESTYS PRISON Onley Rugby Warwickshire C...
3,FLAT 2 RC PRESBYTERY 9 CHAPEL STREET,FLAT 2 ROMAN CATHOLIC PRESBYTERY 9 CHAPEL STREET


## Why these cases — and the risk to weigh

**Why it helps:** matching is token/term-frequency driven. If a messy record says
`RC CHURCH` but the canonical says `ROMAN CATHOLIC CHURCH` (or vice-versa), the shared
signal is weak. Expanding both sides to a common form makes the tokens line up.

**The risk — over-expansion.** Because expansion is unconditional and single-token,
**every** standalone `RC` becomes `ROMAN CATHOLIC`, and **every** `HMP` becomes
`HIS MAJESTYS PRISON`, regardless of context. For `HMP` that is almost always safe (it
is an unusual token). For `RC` it is worth checking it doesn't routinely appear as
something else (initials, a road/block code, etc.). The single-token architecture means
you **cannot** scope it to "only when followed by CHURCH" without changing the mechanism.

## Proposed contribution

1. Add to `uk_address_matcher/data/address_abbreviations.json`:
   ```json
   { "token": "RC",  "replacement": "ROMAN CATHOLIC" },
   { "token": "HMP", "replacement": "HIS MAJESTYS PRISON" }
   ```
2. Add a test (the project tests cleaning output) asserting that
   `RC CHURCH → ROMAN CATHOLIC CHURCH` and `HMP LEEDS → HIS MAJESTYS PRISON LEEDS`
   after `prepare_data_for_matching`.
3. In the PR, note the over-expansion trade-off for `RC` and (if needed) propose a
   follow-up if multi-word *keys* are ever required — that would mean extending the
   normalisation step beyond a single-token MAP (e.g. an ordered phrase-replacement
   pass), which is a larger change than this data-only addition.